# 03 — GAN d'inpainting

Générateur U-Net (image trouée + masque + bruit) et discriminateur PatchGAN.
À lancer sur Kaggle avec un **GPU** (Settings → Accelerator → GPU T4/P100) et **Internet activé** (pour le `git clone`).
Données d'entrée : les fichiers `zellige_128.npz` et `masques_test_128.npz` produits par le notebook 01,
ajoutés au notebook via « Add Input ».

## 1. Récupération du code et des données

In [ ]:
import os, sys, glob

REPO = "/kaggle/working/inpainting-patrimoine"
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/ayakorchi1/inpainting-patrimoine.git {REPO}
sys.path.insert(0, REPO)
os.chdir(REPO)

import numpy as np, torch
from src.data import trouver_npz

NPZ = trouver_npz("zellige_128.npz")
NPZ_MASQUES = trouver_npz("masques_test_128.npz")
SORTIE = "/kaggle/working"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Données :", NPZ)
print("Masques :", NPZ_MASQUES)
print("Appareil :", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

## 2. Entraînement (hyperparamètres dans `configs/gan.yaml`)

In [ ]:
from src.train_gan import entrainer

historique, infos = entrainer("configs/gan.yaml", NPZ, dossier=SORTIE)
infos

## 3. Courbes des pertes

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

h = pd.read_csv(f"{SORTIE}/historique_gan.csv")
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(h.epoch, h.perte_D); axes[0].set_title("Perte du discriminateur (hinge)")
axes[1].plot(h.epoch, h.adverse); axes[1].set_title("Perte adverse du générateur")
axes[2].plot(h.epoch, h.l1); axes[2].set_title("L1 dans le trou")
v = h.dropna(subset=["psnr_val"])
axes[3].plot(v.epoch, v.psnr_val, marker="o"); axes[3].set_title("PSNR validation (dB)")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_courbes.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Évaluation sur le test (143 images × 9 masques fixes)
Même protocole que la baseline : mêmes masques, mêmes fonctions `psnr_trou` et `ssim_img`.
Bruit fixe (graine 0) pour que le résultat soit reproductible.

In [ ]:
import time
from src.models.gan import Generateur
from src.train_gan import completer
from src.metrics import psnr_trou, ssim_img

ckpt = torch.load(f"{SORTIE}/gan_meilleur.pt", map_location=device, weights_only=False)
G = Generateur(ckpt["config"]["modele"]["base_generateur"]).to(device)
G.load_state_dict(ckpt["generateur"]); G.eval()
print(f"Meilleur modèle : epoch {ckpt['epoch']}, PSNR val {ckpt['psnr_val']:.2f} dB")

X_test = np.load(NPZ)["test"]
M = dict(np.load(NPZ_MASQUES))

lignes = []
for nom, masques in M.items():
    type_, pct = nom.rsplit("_", 1)
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    R = completer(G, X_test, masques, graine=0, device=device)
    if device == "cuda": torch.cuda.synchronize()
    temps_ms = (time.time() - t0) / len(X_test) * 1000
    ps = [psnr_trou(o, r, m) for o, r, m in zip(X_test, R, masques)]
    ss = [ssim_img(o, r) for o, r in zip(X_test, R)]
    lignes.append(dict(methode="gan", type=type_, trou=int(pct),
                       PSNR=np.mean(ps), SSIM=np.mean(ss), temps_ms=temps_ms))

res = pd.DataFrame(lignes)
res.to_csv(f"{SORTIE}/resultats_gan.csv", index=False)
print("PSNR moyen (dB) :")
display(res.pivot_table(index="type", columns="trou", values="PSNR").round(2))
print("SSIM moyen :")
display(res.pivot_table(index="type", columns="trou", values="SSIM").round(3))
print(f"Temps moyen : {res.temps_ms.mean():.2f} ms / image")

# Comparaison avec la baseline (fichier du dépôt)
base = pd.read_csv("results/resultats_baseline.csv")
comp = pd.concat([base, res]).pivot_table(index=["methode", "type"], columns="trou", values="PSNR").round(2)
print("Comparaison PSNR (dB) :"); display(comp)

## 5. Diversité : 4 propositions par image
Pour chaque image, on génère 4 complétions avec 4 bruits différents, puis on mesure l'écart moyen
entre elles dans le trou (échelle 0-255). **0 = les 4 propositions sont identiques.**

In [ ]:
from src.metrics import diversite_trou

GRAINES = [0, 1000, 2000, 3000]
lignes_div = []
for nom, masques in M.items():
    type_, pct = nom.rsplit("_", 1)
    props = [completer(G, X_test, masques, graine=g, device=device) for g in GRAINES]
    div = [diversite_trou([p[i] for p in props], masques[i]) for i in range(len(X_test))]
    lignes_div.append(dict(methode="gan", type=type_, trou=int(pct), diversite=np.mean(div)))

div = pd.DataFrame(lignes_div)
div.to_csv(f"{SORTIE}/resultats_gan_diversite.csv", index=False)
display(div.pivot_table(index="type", columns="trou", values="diversite").round(2))
if div.diversite.mean() < 1:
    print("⚠ Diversité quasi nulle : le générateur ignore le bruit (effondrement de modes).")

## 6. Figures

In [ ]:
import cv2

i = 5   # même image que pour la baseline
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, pct in enumerate([10, 20, 40]):
    m = M[f"central_{pct}"][i]
    abimee = X_test[i].copy(); abimee[m == 1] = 255
    telea = cv2.inpaint(X_test[i], m, 3, cv2.INPAINT_TELEA)
    gan = completer(G, X_test[i:i+1], m[None], graine=0, device=device)[0]
    for col, (im, titre) in enumerate([(abimee, f"Abîmée {pct}%"), (telea, "Telea"),
                                       (gan, "GAN"), (X_test[i], "Originale")]):
        axes[row, col].imshow(im); axes[row, col].set_title(titre); axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_exemples.png", dpi=150, bbox_inches="tight"); plt.show()

# 4 propositions pour une même image (trou central 40 %)
m = M["central_40"][i]
abimee = X_test[i].copy(); abimee[m == 1] = 255
props = [completer(G, X_test[i:i+1], m[None], graine=g, device=device)[0] for g in GRAINES]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
for ax, (im, titre) in zip(axes, [(abimee, "Abîmée 40%")] +
                           [(p, f"Proposition {k+1}") for k, p in enumerate(props)] +
                           [(X_test[i], "Originale")]):
    ax.imshow(im); ax.set_title(titre); ax.axis("off")
plt.tight_layout()
plt.savefig(f"{SORTIE}/gan_propositions.png", dpi=150, bbox_inches="tight"); plt.show()

print("À télécharger depuis /kaggle/working puis à mettre dans results/ du dépôt :")
print("resultats_gan.csv, resultats_gan_diversite.csv, historique_gan.csv, memoire_gpu.json,")
print("gan_courbes.png, gan_exemples.png, gan_propositions.png (PAS les fichiers .pt)")